<a href="http://landlab.github.io"><img style="float: left; width: 300px;" src="https://landlab.csdms.io/_static/landlab_logo.png"></a>

# 2D Surface Water Flow: HLLC Validation — Stoker Dam Break (Wet/Wet)

<hr>
<small>For more Landlab tutorials, click here: <a href="https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html">https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html</a></small>
<hr>

## Overview

This notebook demonstrates the validation of the `RiverFlowDynamics_HLLC` component using a **1D flat-bed dam break with a wet downstream region** ($h_R > 0$). 

This exercises the wet/wet branch of the HLLC Riemann solver. It is the canonical test to verify that the numerical scheme accurately captures both the correct shock-side star state and the rarefaction structure without generating spurious oscillations.

### Theory

The analytical solution was derived by Stoker (1957). The Riemann problem has the structure:
$$h_L \quad | \quad \text{rarefaction} \quad | \quad h_* \quad | \quad \text{shock} \quad | \quad h_R$$

The intermediate depth $h_*$ is constant and is separated from the initial left state $h_L$ by a left-going rarefaction fan, and from the right state $h_R$ by a right-going shock. $h_*$ is found by solving the non-linear compatibility equation balancing the rarefaction Riemann invariant against the shock Rankine-Hugoniot condition:

$$u_L + 2\sqrt{gh_L} = 2\sqrt{gh_*} + (h_* - h_R)\sqrt{\frac{g(h_* + h_R)}{2h_*h_R}}$$

### Import the needed libraries:

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
from scipy.optimize import brentq

from landlab import RasterModelGrid
from landlab.components import RiverFlowDynamics_HLLC

## 1. Define Simulation Parameters

We configure a $1000$ m domain with a strong 10:1 depth ratio ($h_L = 1.0$ m, $h_R = 0.1$ m) to ensure a strong shock wave.

In [ ]:
g_acc = 9.81
h_L = 1.0  # upstream reservoir depth [m]
h_R = 0.1  # downstream depth [m] (10% of h_L → strong shock)
u_L = 0.0
u_R = 0.0
L = 1000.0
dx = 1.0
x_dam = L / 2.0
t_end = 20.0
W = 5 * dx  # Width for 2D grid compliance

c_L = np.sqrt(g_acc * h_L)
c_R = np.sqrt(g_acc * h_R)

print("=" * 72)
print("  Stoker dam break (wet/wet)")
print("=" * 72)
print(f"  h_L = {h_L} m,  h_R = {h_R} m,  c_L = {c_L:.4f},  c_R = {c_R:.4f}")
print(f"  dx = {dx} m,  t_end = {t_end} s")

## 2. Generate the Analytical Solution Function

We define the `brentq` root-finding logic to solve for the exact star state ($h_*$ and $u_*$) and build the full theoretical profile for any given time $t$.

In [ ]:
def _stoker_residual(h_star, h_L, h_R, u_L, u_R, g):
    if h_star <= 0.0 or h_star <= h_R:
        return np.inf
    lhs = u_L + 2.0 * np.sqrt(g * h_L) - 2.0 * np.sqrt(g * h_star)
    rhs = u_R + (h_star - h_R) * np.sqrt(g * (h_star + h_R) / (2.0 * h_star * h_R))
    return lhs - rhs


def stoker_star_state(h_L, h_R, u_L, u_R, g):
    def f(hs):
        return _stoker_residual(hs, h_L, h_R, u_L, u_R, g)

    lo, hi = h_R * (1.0 + 1.0e-9), h_L
    while f(hi) > 0 and hi < 10.0 * h_L:
        hi *= 2.0
    h_star = brentq(f, lo, hi, xtol=1.0e-12)
    u_star = u_L + 2.0 * np.sqrt(g * h_L) - 2.0 * np.sqrt(g * h_star)
    S_shock = (h_star * u_star - h_R * u_R) / (h_star - h_R)
    return h_star, u_star, S_shock


def stoker_profile(x, t, h_L, h_R, u_L, u_R, g, x_dam=0.0):
    h_star, u_star, S_shock = stoker_star_state(h_L, h_R, u_L, u_R, g)
    c_L = np.sqrt(g * h_L)
    c_star = np.sqrt(g * h_star)

    x_rare_head = x_dam + (u_L - c_L) * t
    x_rare_tail = x_dam + (u_star - c_star) * t
    x_shock = x_dam + S_shock * t

    h = np.empty_like(x)
    u = np.empty_like(x)
    for i, xi in enumerate(x):
        if xi <= x_rare_head:
            h[i], u[i] = h_L, u_L
        elif xi <= x_rare_tail:
            xi_norm = (xi - x_dam) / t
            u[i] = (2.0 / 3.0) * (xi_norm + 0.5 * u_L + c_L)
            c = (1.0 / 3.0) * (-xi_norm + u_L + 2.0 * c_L)
            h[i] = c * c / g
        elif xi <= x_shock:
            h[i], u[i] = h_star, u_star
        else:
            h[i], u[i] = h_R, u_R
    return h, u, (h_star, u_star, S_shock, x_rare_head, x_rare_tail, x_shock)


h_star_anal, u_star_anal, S_shock_anal = stoker_star_state(h_L, h_R, u_L, u_R, g_acc)
print(
    f"Analytical Star State:\n  h* = {h_star_anal:.4f} m\n  u* = {u_star_anal:.4f} m/s\n  S_shock = {S_shock_anal:.4f} m/s"
)

## 3. Configure the Grid and Initial Conditions

We map the two-state condition ($h_L$ and $h_R$) onto a flat-bed Landlab grid.

In [ ]:
ncols = int(round(L / dx))
nrows = int(round(W / dx))
grid = RasterModelGrid((nrows, ncols), xy_spacing=dx)

z = grid.add_zeros("topographic__elevation", at="node")
h = grid.add_zeros("surface_water__depth", at="node")
eta = grid.add_zeros("surface_water__elevation", at="node")

x_node = grid.x_of_node
h[x_node < x_dam] = h_L
h[x_node >= x_dam] = h_R
eta[:] = h + z

h_2d = grid.at_node["surface_water__depth"].reshape(nrows, ncols)
x_1d = x_node.reshape(nrows, ncols)[0]

mass_0 = h_2d.sum() * dx * dx
print(f"Grid initialized: {nrows} rows x {ncols} cols (dx = {dx} m)")
print(f"Initial mass in domain: {mass_0:.4f} m³")

## 4. Component Setup and Live Simulation

We initialize the component using `order=1` spatial reconstruction. We will plot both the **Depth ($h$)** and the **Velocity ($u$)** to verify that the intermediate star state perfectly matches the analytical solution across both variables.

In [ ]:
hllc = RiverFlowDynamics_HLLC(
    grid,
    mannings_n=0.0,
    cfl=0.45,
    order=1,
    wall_edges={"bottom", "top"},
)

# The velocity field is now initialized, so we can bind it here
u_2d = grid.at_node["surface_water__x_velocity"].reshape(nrows, ncols)

t0 = time.time()
display_dt = 4.0
next_display_t = 0.0

# Variables to track for calculating the numerical shock speed post-simulation
shock_tracker_times = []
shock_tracker_pos = []
h_mid = 0.5 * (h_star_anal + h_R)

print("Running Simulation...")

while hllc.elapsed_time < t_end - 1e-9:
    hllc.run_one_step()
    t_curr = hllc.elapsed_time

    # Track the shock position continuously
    h_x_mean = h_2d.mean(axis=0)
    above_mid = h_x_mean > h_mid
    if above_mid.any():
        shock_tracker_times.append(t_curr)
        shock_tracker_pos.append(x_1d[above_mid].max())

    # Live Plotting
    if t_curr >= next_display_t or t_curr >= t_end - 1e-9:
        clear_output(wait=True)

        u_x_mean = u_2d.mean(axis=0)
        h_anal, u_anal, refs = stoker_profile(
            x_1d, t_curr, h_L, h_R, u_L, u_R, g_acc, x_dam
        )
        _, _, _, x_rare_head, x_rare_tail, x_shock = refs

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

        # --- Depth Plot ---
        ax1.plot(x_1d, h_x_mean, "b-", lw=2.0, label="HLLC numerical (order=1)")
        ax1.plot(x_1d, h_anal, "k--", lw=1.5, label="Stoker analytical")
        ax1.axhline(
            h_star_anal,
            color="gray",
            ls=":",
            lw=1.5,
            label=f"Analytical $h^*$ = {h_star_anal:.2f} m",
        )
        ax1.axvline(
            x_rare_tail, color="g", ls=":", lw=1.0, alpha=0.6, label="Rarefaction Tail"
        )
        ax1.axvline(x_shock, color="r", ls=":", lw=1.0, alpha=0.6, label="Shock Front")
        ax1.set_ylabel("Depth h [m]")
        ax1.set_xlabel("Distance x [m]")
        ax1.set_xlim(x_dam - 150, x_dam + 200)
        ax1.set_ylim(0, 1.1)
        ax1.set_title(f"Depth Profile | t = {t_curr:.2f} s")
        ax1.grid(alpha=0.3)
        ax1.legend(loc="upper right", fontsize=8, framealpha=0.9)

        # --- Velocity Plot ---
        ax2.plot(x_1d, u_x_mean, "m-", lw=2.0, label="HLLC numerical (order=1)")
        ax2.plot(x_1d, u_anal, "k--", lw=1.5, label="Stoker analytical")
        ax2.axhline(
            u_star_anal,
            color="gray",
            ls=":",
            lw=1.5,
            label=f"Analytical $u^*$ = {u_star_anal:.2f} m/s",
        )
        ax2.axvline(x_rare_tail, color="g", ls=":", lw=1.0, alpha=0.6)
        ax2.axvline(x_shock, color="r", ls=":", lw=1.0, alpha=0.6)
        ax2.set_ylabel("Velocity u [m/s]")
        ax2.set_xlabel("Distance x [m]")
        ax2.set_xlim(x_dam - 150, x_dam + 200)
        ax2.set_ylim(-0.1, 2.5)
        ax2.set_title(f"Velocity Profile | t = {t_curr:.2f} s")
        ax2.grid(alpha=0.3)
        ax2.legend(loc="lower left", fontsize=8, framealpha=0.9)

        plt.tight_layout()
        plt.show()

        next_display_t += display_dt

## 5. Diagnostics & Acceptance Criteria

We isolate the constant middle state between the rarefaction tail and the shock to compute the numerical $h_*$ and $u_*$, and we use the shock tracking history to compute the numerical shock speed $S_{shock}$.

In [ ]:
t_final = hllc.elapsed_time
mass_end = h_2d.sum() * dx * dx
mass_err = abs(mass_end - mass_0) / mass_0

# Final numerical profiles
h_num = h_2d.mean(axis=0)
u_num = u_2d.mean(axis=0)

# Retrieve final analytical boundaries
_, _, refs = stoker_profile(x_1d, t_final, h_L, h_R, u_L, u_R, g_acc, x_dam)
_, _, _, _, x_rt_tail, x_sh = refs

# Extract numerical star state from the safe interior window
margin = 0.05 * (x_sh - x_rt_tail)
window = (x_1d > x_rt_tail + margin) & (x_1d < x_sh - margin)
h_star_num = np.median(h_num[window])
u_star_num = np.median(u_num[window])

# Compute numerical shock speed from tracking history
S_shock_num = (shock_tracker_pos[-1] - shock_tracker_pos[0]) / (
    shock_tracker_times[-1] - shock_tracker_times[0]
)

h_err = abs(h_star_num - h_star_anal) / h_star_anal
u_err = abs(u_star_num - u_star_anal) / abs(u_star_anal)
S_err = abs(S_shock_num - S_shock_anal) / S_shock_anal

print("=" * 72)
print("  Summary")
print("=" * 72)
print(f"  Star-state h* error : {h_err * 100:7.3f} %   (target < 2 %)")
print(f"  Star-state u* error : {u_err * 100:7.3f} %   (target < 2 %)")
print(f"  Shock speed error   : {S_err * 100:7.3f} %   (target < 2 %)")
print(f"  Mass conservation   : {mass_err * 100:9.4f} % (target < 0.01 %)")
print()

if h_err < 0.02 and u_err < 0.02 and S_err < 0.02 and mass_err < 1e-4:
    print("  Status: PASS ✅")
    print("  The wet/wet HLLC branch captures the shock and rarefaction accurately.")
else:
    print("  Status: REVIEW ⚠️")
print("=" * 72)

## Interpretation of Results

This benchmark rigorously tests the core intermediate state calculations of the HLLC Riemann solver on a wet bed.

1. **Star State Agreement:** The numerical plateau perfectly matching the analytical $h_*$ and $u_*$ proves that the fundamental mass and momentum flux exchanges across the cell interfaces are resolving the correct Rankine-Hugoniot jump conditions.
2. **Shock Capturing without Oscillations:** A first-order scheme naturally smears the steepness of the shock and the head/tail of the rarefaction fan due to numerical diffusion. However, notice that it introduces strictly **zero spurious oscillations** (no "ringing" near the shock), ensuring unconditional stability.

-- --
### And that's it!

You have successfully validated the wet/wet Stoker exact Riemann characteristics of the `RiverFlowDynamics_HLLC` component.

-- --

### Click here for more <a href="https://landlab.csdms.io/tutorials/">Landlab tutorials</a>